In [1]:

# Task 3.1 — Step 1: Load cleaned appointments

import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()

df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)
print("Total no-shows:", df["no_show"].sum())
print("Missing values:", df.isnull().sum().sum())

print("\nFirst 5 rows:")
print(df.head().to_string(index=False))


Saving appointments_clean.csv to appointments_clean.csv
Dataset shape: (990, 16)
Total no-shows: 218
Missing values: 0

First 5 rows:
appointment_id patient_id clinic_id  booked_on  appt_date  age gender  distance_km  fee_inr booking_channel   visit_type reminder_sent  prior_no_shows   status  lead_days  no_show
         A0001       P509       C05 2025-01-24 2025-01-25   60      F         18.0      850           Phone Consultation           Yes               1  No-Show          1        1
         A0002       P046       C07 2025-05-27 2025-06-06   39      M         12.5      350         Walk-In    Follow-up            No               2 Attended         10        0
         A0003       P009       C02 2025-03-24 2025-03-30   63      F          4.1      850         Walk-In Consultation            No               0 Attended          6        0
         A0004       P482       C01 2025-04-18 2025-04-22   52      F          7.5      750             App Consultation           Yes            

In [2]:

# Task 3.1 — Step 2: Feature selection and encoding

feature_columns = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type"
]

# Select only the permitted input features
X = df[feature_columns].copy()

# Convert reminder values to numeric
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# One-hot encode categorical features
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target: 1 = No-Show, 0 = Attended
y = df["no_show"].copy()

# Verify the resulting features
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature names:")
print(X.columns.tolist())

print("\nMissing feature values:", X.isnull().sum().sum())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nFirst 5 encoded rows:")
print(X.head().to_string(index=False))


Feature matrix shape: (990, 11)
Target shape: (990,)

Feature names:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Missing feature values: 0

Target distribution:
no_show
0    772
1    218
Name: count, dtype: int64

First 5 encoded rows:
 age  distance_km  fee_inr  lead_days  prior_no_shows  reminder_sent  booking_channel_App  booking_channel_Phone  booking_channel_Walk-In  visit_type_Consultation  visit_type_Follow-up
  60         18.0      850          1               1              1                False                   True                    False                     True                 False
  39         12.5      350         10               2              0                False                  False                     True                    False                  True
  63          4.1      850          6       

In [3]:

# Task 3.1 — Step 3: Train-test split and baseline

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split features and target
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Baseline: predict every appointment as Attended (0)
baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

# Verify results
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining no-shows:", y_train.sum())
print("Testing no-shows:", y_test.sum())

print("\nBaseline accuracy:", round(baseline_accuracy, 4))
print("Baseline accuracy (%):", round(baseline_accuracy * 100, 2))


Training rows: 792
Testing rows: 198

Training no-shows: 169
Testing no-shows: 49

Baseline accuracy: 0.7525
Baseline accuracy (%): 75.25


In [4]:

# Task 3.2 — Step 1: Scale features and train Logistic Regression

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create the scaler
scaler = StandardScaler()

# Fit scaler ONLY on training data
X_train_s = scaler.fit_transform(X_train)

# Apply the same scaling to test data
X_test_s = scaler.transform(X_test)

# Create and train Logistic Regression
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_s, y_train)

# Verify training
print("Model trained successfully!")
print("Training data shape:", X_train_s.shape)
print("Testing data shape:", X_test_s.shape)
print("Number of model coefficients:", len(lr_model.coef_[0]))
print("Model classes:", lr_model.classes_)


Model trained successfully!
Training data shape: (792, 11)
Testing data shape: (198, 11)
Number of model coefficients: 11
Model classes: [0 1]


In [5]:

# Task 3.2 — Step 2: Evaluate Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predict appointment outcomes
lr_predictions = lr_model.predict(X_test_s)

# Predict probabilities of no-show (class 1)
lr_probabilities = lr_model.predict_proba(X_test_s)[:, 1]

# Calculate evaluation metrics
lr_accuracy = accuracy_score(y_test, lr_predictions)
lr_precision = precision_score(y_test, lr_predictions)
lr_recall = recall_score(y_test, lr_predictions)
lr_f1 = f1_score(y_test, lr_predictions)
lr_auc = roc_auc_score(y_test, lr_probabilities)

# Display results
print("Logistic Regression Evaluation")
print("-" * 35)

print("Accuracy:", round(lr_accuracy, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))

print("\nPrecision:", round(lr_precision, 4))
print("Recall:", round(lr_recall, 4))
print("F1-score:", round(lr_f1, 4))
print("ROC-AUC:", round(lr_auc, 4))


Logistic Regression Evaluation
-----------------------------------
Accuracy: 0.7879

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.6296
Recall: 0.3469
F1-score: 0.4474
ROC-AUC: 0.7889


In [6]:

# Task 3.2 — Step 3: Coefficients and reminder odds ratios

# Display coefficients for all features
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr_model.coef_[0]
})

print("Logistic Regression Coefficients:")
print(coefficients.to_string(index=False))

# Find the reminder feature's position
reminder_index = X.columns.get_loc("reminder_sent")

# Get its coefficient and standard deviation
reminder_coef = lr_model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

# Calculate both odds ratios
scaled_odds_ratio = np.exp(reminder_coef)

actual_odds_ratio = np.exp(
    reminder_coef / reminder_scale
)

print("\nReminder Sent Analysis")
print("-" * 35)
print("Reminder coefficient:", round(reminder_coef, 4))
print("Reminder standard deviation:", round(reminder_scale, 4))
print("Scaled odds ratio:", round(scaled_odds_ratio, 4))
print("No-to-Yes odds ratio:", round(actual_odds_ratio, 4))


Logistic Regression Coefficients:
                feature  coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658

Reminder Sent Analysis
-----------------------------------
Reminder coefficient: -0.5807
Reminder standard deviation: 0.495
Scaled odds ratio: 0.5595
No-to-Yes odds ratio: 0.3094


In [7]:

# Task 3.3 — Decision Trees and Overfitting

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Model 1: Unrestricted Decision Tree
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Model 2: Decision Tree with max_depth=3
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_depth3.fit(X_train, y_train)

# Calculate training and test accuracy
full_train_acc = accuracy_score(
    y_train, tree_full.predict(X_train)
)
full_test_acc = accuracy_score(
    y_test, tree_full.predict(X_test)
)

depth3_train_acc = accuracy_score(
    y_train, tree_depth3.predict(X_train)
)
depth3_test_acc = accuracy_score(
    y_test, tree_depth3.predict(X_test)
)

print("Decision Tree Comparison")
print("-" * 40)

print("Unrestricted Decision Tree:")
print("Training accuracy:", round(full_train_acc, 4))
print("Test accuracy:", round(full_test_acc, 4))

print("\nDecision Tree (max_depth=3):")
print("Training accuracy:", round(depth3_train_acc, 4))
print("Test accuracy:", round(depth3_test_acc, 4))

print("\nAccuracy gaps:")
print("Unrestricted tree:",
      round(full_train_acc - full_test_acc, 4))
print("Depth-3 tree:",
      round(depth3_train_acc - depth3_test_acc, 4))


Decision Tree Comparison
----------------------------------------
Unrestricted Decision Tree:
Training accuracy: 1.0
Test accuracy: 0.6717

Decision Tree (max_depth=3):
Training accuracy: 0.822
Test accuracy: 0.7778

Accuracy gaps:
Unrestricted tree: 0.3283
Depth-3 tree: 0.0442


In [8]:

# Task 3.4 — Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

# Predict test outcomes and probabilities
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_precision = precision_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_f1 = f1_score(y_test, rf_predictions)
rf_auc = roc_auc_score(y_test, rf_probabilities)

print("Random Forest Evaluation")
print("-" * 35)

print("Accuracy:", round(rf_accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))

print("\nPrecision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1-score:", round(rf_f1, 4))
print("ROC-AUC:", round(rf_auc, 4))

# Display feature importance
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

print("\nRandom Forest Feature Importances:")
print(feature_importance.to_string(index=False))


Random Forest Evaluation
-----------------------------------
Accuracy: 0.7475

Confusion Matrix:
[[133  16]
 [ 34  15]]

Precision: 0.4839
Recall: 0.3061
F1-score: 0.375
ROC-AUC: 0.7057

Random Forest Feature Importances:
                feature  importance
              lead_days    0.219905
                    age    0.196788
            distance_km    0.194739
                fee_inr    0.137400
         prior_no_shows    0.105709
          reminder_sent    0.045711
  booking_channel_Phone    0.026913
booking_channel_Walk-In    0.019735
visit_type_Consultation    0.019364
    booking_channel_App    0.018394
   visit_type_Follow-up    0.015341


In [11]:

# Task 3.4 — Model comparison table

# Calculate missing depth-3 tree metrics
depth3_predictions = tree_depth3.predict(X_test)
depth3_probabilities = tree_depth3.predict_proba(X_test)[:, 1]

depth3_recall = recall_score(y_test, depth3_predictions)
depth3_auc = roc_auc_score(y_test, depth3_probabilities)

# Build comparison table
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest"
    ],
    "Accuracy": [
        lr_accuracy,
        depth3_test_acc,
        rf_accuracy
    ],
    "Recall": [
        lr_recall,
        depth3_recall,
        rf_recall
    ],
    "ROC-AUC": [
        lr_auc,
        depth3_auc,
        rf_auc
    ]
})

print("Task 3.4 — Model Comparison")
print("-" * 50)
print(model_comparison.round(4).to_string(index=False))


Task 3.4 — Model Comparison
--------------------------------------------------
                  Model  Accuracy  Recall  ROC-AUC
    Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (Depth 3)    0.7778  0.2245   0.6997
          Random Forest    0.7475  0.3061   0.7057


In [12]:

# Task 3.5(a) — Random Forest Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

# Fit using TRAINING DATA ONLY
grid_search.fit(X_train, y_train)

# Display the best parameters and cross-validated score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validated ROC-AUC:",
      round(grid_search.best_score_, 4))


Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validated ROC-AUC: 0.7673


In [13]:

# Task 3.5(b) — Compare models using training-only CV

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Logistic Regression pipeline:
# Scaling happens separately within each CV fold
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Calculate mean 5-fold ROC-AUC for each model
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = grid_search.best_score_

# Compare results
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV ROC-AUC": [
        lr_cv_auc,
        tree_cv_auc,
        rf_cv_auc,
        tuned_rf_cv_auc
    ]
}).sort_values(
    by="CV ROC-AUC",
    ascending=False
)

print("Training Cross-Validation Comparison")
print("-" * 50)
print(cv_comparison.round(4).to_string(index=False))

print("\nBest model:", cv_comparison.iloc[0]["Model"])


Training Cross-Validation Comparison
--------------------------------------------------
                  Model  CV ROC-AUC
    Logistic Regression      0.7737
    Tuned Random Forest      0.7673
          Random Forest      0.7466
Decision Tree (Depth 3)      0.6784

Best model: Logistic Regression


In [14]:

# Task 3.5(c) — Threshold selection using training-only predictions

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Generate out-of-fold probabilities for Logistic Regression
oof_probabilities = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare the required thresholds
threshold_results = []

for threshold in [0.50, 0.35]:

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, predictions),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions),
        "Flagged": predictions.sum()
    })

threshold_comparison = pd.DataFrame(threshold_results)

print("Training Threshold Comparison")
print("-" * 55)
print(
    threshold_comparison.round(4).to_string(index=False)
)

# Select the threshold with the higher F1-score
best_row = threshold_comparison.loc[
    threshold_comparison["F1"].idxmax()
]

chosen_threshold = float(best_row["Threshold"])

print("\nChosen threshold:", chosen_threshold)
print("Reason: Higher out-of-fold training F1-score")


Training Threshold Comparison
-------------------------------------------------------
 Threshold  Precision  Recall     F1  Flagged
      0.50     0.5753  0.2485 0.3471       73
      0.35     0.5130  0.4675 0.4892      154

Chosen threshold: 0.35
Reason: Higher out-of-fold training F1-score


In [15]:

# Task 3.5(d) — Final model evaluation

# Fit the selected pipeline on all training data
lr_pipe.fit(X_train, y_train)

# Predict no-show probabilities on the test set
final_probabilities = lr_pipe.predict_proba(X_test)[:, 1]

# Apply the selected threshold of 0.35
final_predictions = (
    final_probabilities >= chosen_threshold
).astype(int)

# Calculate final evaluation metrics
final_precision = precision_score(y_test, final_predictions)
final_recall = recall_score(y_test, final_predictions)
final_auc = roc_auc_score(y_test, final_probabilities)
final_flagged = int(final_predictions.sum())

print("Final Model Evaluation")
print("-" * 40)

print("Selected model: Logistic Regression")
print("Selected threshold:", chosen_threshold)

print("\nTest precision:", round(final_precision, 4))
print("Test recall:", round(final_recall, 4))
print("Test ROC-AUC:", round(final_auc, 4))
print("Appointments flagged:", final_flagged)
print("Total test appointments:", len(y_test))


Final Model Evaluation
----------------------------------------
Selected model: Logistic Regression
Selected threshold: 0.35

Test precision: 0.5714
Test recall: 0.5714
Test ROC-AUC: 0.7889
Appointments flagged: 49
Total test appointments: 198


In [16]:

# Task 3.5(e) — Export test risk scores

# Create a DataFrame for test appointments
risk_scores_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_probabilities,
    "flag": final_predictions
})

# Save the required CSV file
risk_scores_df.to_csv(
    "test_risk_scores.csv",
    index=False
)

# Verify the exported file
saved_risk_scores = pd.read_csv("test_risk_scores.csv")

print("Saved file: test_risk_scores.csv")
print("Dataset shape:", saved_risk_scores.shape)
print("Column names:", saved_risk_scores.columns.tolist())
print("Missing values:", saved_risk_scores.isnull().sum().sum())
print("Unique appointment IDs:",
      saved_risk_scores["appointment_id"].nunique())
print("Flagged appointments:",
      saved_risk_scores["flag"].sum())

print("\nFirst 5 rows:")
print(saved_risk_scores.head().to_string(index=False))


Saved file: test_risk_scores.csv
Dataset shape: (198, 4)
Column names: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Missing values: 0
Unique appointment IDs: 198
Flagged appointments: 49

First 5 rows:
appointment_id  no_show  risk_prob  flag
         A0217        0   0.021565     0
         A0336        0   0.150917     0
         A0508        0   0.518692     1
         A0314        0   0.191300     0
         A0089        0   0.167718     0
